# 🚀 GEMMA 3N E4B-IT TURKISH TELCO TRAINING

## Using Unsloth for 2x faster training with 50% less memory

This notebook fine-tunes **Gemma 3N E4B-IT 4-bit** for Turkish telco call center.

In [ ]:
%%capture
# Install Unsloth
import torch
major_version, minor_version = torch.cuda.get_device_capability()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
if major_version >= 8:
    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
else:
    !pip install --no-deps xformers trl peft accelerate bitsandbytes
pass

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto detect
load_in_4bit = True  # Use 4bit quantization

# Load GEMMA 3N E4B-IT
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it-bnb-4bit",  # Closest supported model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("✅ Model loaded!")

In [ ]:
# Upload training data
from google.colab import files
print("📤 Please upload gemma3n_training.jsonl")
uploaded = files.upload()

# Load data
import json
training_data = []
with open('gemma3n_training.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            training_data.append(json.loads(line))

print(f"✅ Loaded {len(training_data)} examples")

In [ ]:
# Setup LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,  # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0.1,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 4x longer context
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

print("✅ LoRA configured!")

In [ ]:
# Prepare dataset
from datasets import Dataset

# Format for training
formatted_data = []
for item in training_data:
    text = item.get('text', '')
    if text:
        formatted_data.append({'text': text, 'output': ''})

dataset = Dataset.from_list(formatted_data)
print(f"✅ Dataset ready: {len(dataset)} examples")

In [ ]:
# Training
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

print("🚀 Starting training...")

In [ ]:
# Train!
trainer_stats = trainer.train()
print("✅ Training complete!")

In [ ]:
# Test inference
FastLanguageModel.for_inference(model)

test_prompts = [
    "eSIM'im çalışmıyor, yardım eder misiniz?",
    "Faturamda haksız ücret var!",
    "Daha hızlı internet paketi istiyorum"
]

for prompt in test_prompts:
    formatted = f"""### Instruction:
Sen bir Türk telekom asistanısın.

### Input:
[DUYGU: normal]
{prompt}

### Output:"""
    
    inputs = tokenizer([formatted], return_tensors="pt").to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True)
    response = tokenizer.batch_decode(outputs)
    
    print(f"\n📞 Customer: {prompt}")
    print(f"🤖 Agent: {response[0].split('### Output:')[-1].strip()[:200]}")
    print("-" * 50)

In [ ]:
# Save model
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# Save to 16bit for better compatibility
model.save_pretrained_merged("model", tokenizer, save_method="merged_16bit")

print("✅ Model saved!")

In [ ]:
# Download model
from google.colab import files
import shutil

# Zip the model
shutil.make_archive('gemma3n_telco_finetuned', 'zip', 'model')
files.download('gemma3n_telco_finetuned.zip')

print("✅ Model downloaded! Ready for deployment!")